# C9orf72 STR Genotyping from UK Biobank DRAGEN WGS

This notebook extracts C9orf72 repeat counts (REPCN) from individual STR VCF files in UK Biobank.

**Pipeline:**
1. Install bcftools via conda
2. Build a VCF manifest from the UKB directory listing
3. Query each sample's VCF in parallel for the C9orf72 region

**Output:** `c9orf72_genotypes_results.tsv` — one row per sample with eid and REPCN.

## Step 1: Install bcftools

In [ ]:
# # Build up env, only run once
# conda update -n base -c defaults conda -y
# conda config --add channels conda-forge
# conda config --add channels bioconda
# conda config --add channels defaults
# conda config --set channel_priority strict
# conda create -y -n bioinfo -c conda-forge -c bioconda bcftools

In [ ]:
conda activate bioinfo

In [ ]:
bcftools --version | head -1

## Step 2: Build VCF manifest

List all STR VCF files from UKB directories 10–60, then parse into a TSV manifest.

This replaces the separate R notebook — the same parsing is done with awk.

In [ ]:
# --- Configuration ---
BASE_DIR="Bulk/DRAGEN WGS/Whole genome STR call files (DRAGEN) [500k release]/"
RAW_LIST_FILE="dx_ls_raw_output.txt"
MANIFEST_FILE="vcf_manifest.tsv"

# List all files in directories 10-60
> "${RAW_LIST_FILE}"
for i in {10..60}; do
    subdir="${BASE_DIR}${i}/"
    dx ls "${subdir}" 2>/dev/null | while read -r f; do
        echo "${subdir}${f}"
    done >> "${RAW_LIST_FILE}" || true
done
echo "Raw listing: $(wc -l < ${RAW_LIST_FILE}) files"

In [ ]:
# Parse raw listing into a clean manifest (replaces the R notebook)
echo -e "vcf_path\tfilename\tid\teid" > "${MANIFEST_FILE}"

grep '\.vcf\.gz$' "${RAW_LIST_FILE}" | awk -F'/' '{
    path = $0
    filename = $NF
    # id = everything before first dot
    split(filename, a, ".")
    id = a[1]
    # eid = everything before first underscore
    split(id, b, "_")
    eid = b[1]
    print path "\t" filename "\t" id "\t" eid
}' | sort -t$'\t' -k3,3 -u >> "${MANIFEST_FILE}"

echo "Manifest: $(tail -n +2 ${MANIFEST_FILE} | wc -l) unique samples"
head -3 "${MANIFEST_FILE}"

## Step 3: Debug — test one sample before running all

In [ ]:
# Pick the first sample and download it for testing
BASE_DIR="Bulk/DRAGEN WGS/Whole genome STR call files (DRAGEN) [500k release]/"
RAW_LIST_FILE="dx_ls_raw_output.txt"
MANIFEST_FILE="vcf_manifest.tsv"

TEST_LINE=$(head -2 ${MANIFEST_FILE} | tail -1)
TEST_PATH=$(echo "$TEST_LINE" | cut -f1)
TEST_FILE=$(echo "$TEST_LINE" | cut -f2)
TEST_EID=$(echo "$TEST_LINE" | cut -f4)
TEST_TBI_PATH="$(dirname ${TEST_PATH})/${TEST_FILE}.tbi"

echo "Testing eid: ${TEST_EID}"
echo "Downloading: ${TEST_PATH}"
dx download "${TEST_PATH}" -o "${TEST_FILE}" -f
dx download "${TEST_TBI_PATH}" -o "${TEST_FILE}.tbi" -f
echo "Downloaded: $(ls -lh ${TEST_FILE} ${TEST_FILE}.tbi)"
echo ""

# Show what chromosomes/contigs are in the VCF header
echo "=== Contigs in VCF ==="
bcftools view -h "${TEST_FILE}" 2>&1 | grep "^##contig" | head -5
echo ""

# Try region query WITH stderr visible
echo "=== Region query chr9:27573525-27573555 ==="
bcftools query -r "chr9:27573525-27573555" -f '%CHROM\t%POS\t%INFO/REPID\t[%REPCN]\n' "${TEST_FILE}" 2>&1
echo ""

# Try without chr prefix
echo "=== Region query 9:27573525-27573555 ==="
bcftools query -r "9:27573525-27573555" -f '%CHROM\t%POS\t%INFO/REPID\t[%REPCN]\n' "${TEST_FILE}" 2>&1
echo ""

# Search for C9orf72 by REPID across the whole file (no region filter)
echo "=== Grep for C9orf72 in entire VCF ==="
bcftools query -f '%CHROM\t%POS\t%INFO/REPID\t[%REPCN]\n' "${TEST_FILE}" 2>&1 | grep -i "c9orf72"
echo ""

# Show a few lines around chr9 to see what format looks like
echo "=== Sample chr9 entries ==="
bcftools query -f '%CHROM\t%POS\t%INFO/REPID\t[%REPCN]\n' "${TEST_FILE}" 2>&1 | grep -E "^chr9\t|^9\t" | head -5
echo ""

# Cleanup
rm -f "${TEST_FILE}" "${TEST_FILE}.tbi"

## Step 4: Extract C9orf72 REPCN in parallel

**Run this after confirming the correct region/field from the debug cell above.**

Update REGION and the bcftools format string if needed.

In [ ]:
# --- Configuration (update REGION based on debug output above) ---
RESULTS_FILE="c9orf72_genotypes_results.tsv"
REGION="chr9:27573525-27573555"
PARALLEL_THREADS=100

# Header
echo -e "eid\tc9orf72_REPCN" > "${RESULTS_FILE}"

# Worker function: dx download VCF+TBI, query REPCN, cleanup
process_sample() {
    local line="$1"
    IFS=$'\t' read -r vcf_path filename _id eid <<< "$line"

    local tbi_path="$(dirname "${vcf_path}")/${filename}.tbi"

    # Download VCF + index
    dx download "${vcf_path}" -o "${filename}" > /dev/null 2>&1
    dx download "${tbi_path}" -o "${filename}.tbi" > /dev/null 2>&1

    # Query
    local genotype
    genotype=$(bcftools query -r "${REGION}" -f '[%REPCN]' "${filename}" 2>/dev/null)
    [[ $? -ne 0 || -z "${genotype}" ]] && genotype="not_found"

    # Append result & cleanup
    echo -e "${eid}\t${genotype}" >> "${RESULTS_FILE}"
    rm -f "${filename}" "${filename}.tbi"
}

export -f process_sample
export REGION RESULTS_FILE

# Run in parallel
tail -n +2 "${MANIFEST_FILE}" | xargs -d '\n' -P "${PARALLEL_THREADS}" -I {} bash -c 'process_sample "{}"'

echo "Done! Results: $(wc -l < ${RESULTS_FILE}) lines (including header)"

In [ ]:
# Quick check
head "${RESULTS_FILE}"
echo "---"
echo "Total samples: $(tail -n +2 ${RESULTS_FILE} | wc -l)"
echo "Not found: $(grep -c 'not_found' ${RESULTS_FILE})"